# YOLOv11l-seg Industrial Defect Segmentation — Colab GPU Runner (EXP-00 to EXP-08)

Runs the **9-configuration strict defect-free ablation protocol** (`YOLOv11l-seg@640`, 100 epochs, seed 42) across 6 representative MVTec AD categories:
- **3 Periodic Textures**: `grid`, `carpet`, `tile`
- **3 Homogeneous Objects**: `screw`, `pill`, `bottle`

**Flexible Execution + Automatic Persistence**:
- Each experiment (`EXP-00` to `EXP-08`) has its **own dedicated runnable cell** AND supports batch filtering via `RUN_ONLY`.
- After every experiment completes, `summary.json`, `robustness/` (for `EXP-01` & `EXP-05`), and `best.pt` are synced immediately to Google Drive (`/content/drive/MyDrive/industrial-defect-segmentation/`), with automatic resume if Colab disconnects.

## 1. Mount Google Drive & Install Dependencies

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/industrial-defect-segmentation'
DRIVE_RESULTS = os.path.join(DRIVE_DIR, 'results')
DRIVE_CKPTS = os.path.join(DRIVE_DIR, 'checkpoints')
DRIVE_DATA = os.path.join(DRIVE_DIR, 'mvtec_ad')

for d in (DRIVE_DIR, DRIVE_RESULTS, DRIVE_CKPTS, DRIVE_DATA):
    os.makedirs(d, exist_ok=True)

print(f'Persistent Drive workspace ready at: {DRIVE_DIR}')

In [ ]:
!pip install -q ultralytics pywavelets scipy scikit-learn tabulate huggingface_hub pandas pyyaml
!nvidia-smi

## 2. Clone / Pull Latest Code from GitHub (`origin/main`)

In [ ]:
import os
REPO_URL = 'https://github.com/nghiemmmm/industrial-defect-segmentation.git'
REPO_DIR = '/content/industrial-defect-segmentation'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin main

%cd {REPO_DIR}
!git log -1 --oneline
!pytest tests/ -q

## 3. Prepare MVTec AD Dataset on Fast Local SSD

In [ ]:
import os
ARCHIVE_PATH = os.path.join(DRIVE_DATA, 'mvtec_ad_6cat.tar.gz')

if os.path.exists('mvtec_ad/bottle') and os.path.exists('mvtec_ad/grid'):
    print('MVTec AD 6 categories already present on local SSD.')
elif os.path.exists(ARCHIVE_PATH):
    print(f'Extracting {ARCHIVE_PATH} to local SSD...')
    !tar -xzf "{ARCHIVE_PATH}" -C .
else:
    print('Downloading 6 MVTec AD categories via CDN...')
    !python tools/download_mvtec_categories.py

!ls -la mvtec_ad

## 4. Runner Helper & Optional Filter (`RUN_ONLY`)
- Leave `RUN_ONLY = None` to execute all 9 cells sequentially (skipping any experiment already completed in Drive).
- Or set e.g. `RUN_ONLY = ['exp00_baseline', 'exp01_cutpaste']` to run only a subset when splitting across Colab sessions.
- You can also click Play on any individual `EXP-xx` cell below directly.

In [ ]:
import os, shutil, subprocess

# Set to None to run all 9 experiments, or list specific experiment IDs to filter:
# Example: RUN_ONLY = ['exp00_baseline', 'exp01_cutpaste', 'exp02_draem']
RUN_ONLY = None

def run_and_sync(exp_id: str, cfg_path: str, force: bool = False):
    if RUN_ONLY is not None and exp_id not in RUN_ONLY:
        print(f'[FILTERED] Skipping {exp_id} (not in RUN_ONLY)')
        return

    drive_exp_dir = os.path.join(DRIVE_RESULTS, exp_id)
    drive_summary = os.path.join(drive_exp_dir, 'summary.json')
    local_exp_dir = f'results/experiments/{exp_id}'

    if not force and os.path.exists(drive_summary):
        print(f'[SKIP] {exp_id} already completed in Drive: {drive_summary}')
        os.makedirs(local_exp_dir, exist_ok=True)
        shutil.copy2(drive_summary, f'{local_exp_dir}/summary.json')
        return

    print(f'\n==================== RUNNING {exp_id.upper()} ====================')
    ret = subprocess.run(['python', 'experiments/run_experiment.py', '--config', cfg_path], check=False)
    if ret.returncode != 0:
        raise RuntimeError(f'{exp_id} failed with exit code {ret.returncode}')

    # Immediately persist summary, robustness curves, and best checkpoint to Drive
    os.makedirs(drive_exp_dir, exist_ok=True)
    shutil.copy2(f'{local_exp_dir}/summary.json', drive_summary)
    rob_src = f'{local_exp_dir}/robustness'
    if os.path.isdir(rob_src):
        shutil.copytree(rob_src, os.path.join(drive_exp_dir, 'robustness'), dirs_exist_ok=True)
    best_pt = f'{local_exp_dir}/train_run/weights/best.pt'
    if os.path.exists(best_pt):
        shutil.copy2(best_pt, os.path.join(DRIVE_CKPTS, f'{exp_id}_best.pt'))
    # Persist per-epoch training loss logs, args, and evaluation curves to Drive
    train_run_dir = f'{local_exp_dir}/train_run'
    if os.path.isdir(train_run_dir):
        logs_dst = os.path.join(drive_exp_dir, 'train_logs')
        os.makedirs(logs_dst, exist_ok=True)
        for fname in (
            'results.csv', 'results.png', 'args.yaml',
            'MaskPR_curve.png', 'MaskF1_curve.png', 'MaskP_curve.png', 'MaskR_curve.png',
            'confusion_matrix.png', 'confusion_matrix_normalized.png',
        ):
            src_f = os.path.join(train_run_dir, fname)
            if os.path.exists(src_f):
                shutil.copy2(src_f, os.path.join(logs_dst, fname))
    # Free temporary exported PNG dataset on Colab SSD after Drive sync succeeds
    shutil.rmtree(f'datasets/processed/{exp_id}', ignore_errors=True)
    print(f'[SYNCED & CLEANED] {exp_id} -> {drive_exp_dir}')

print('Helper run_and_sync() ready. RUN_ONLY =', RUN_ONLY)

## 5. Individual Experiment Cells (EXP-00 to EXP-08, Seed 42)

In [ ]:
# EXP-00: CutPaste (Base), No Freq Filter, rho_neg = 0.00
run_and_sync('exp00_baseline', 'configs/experiments/exp00_baseline.yaml')

In [ ]:
# EXP-01: CutPaste, No Freq Filter, rho_neg = 0.15 + MVTec-AD-C (Table V Baseline)
run_and_sync('exp01_cutpaste', 'configs/experiments/exp01_cutpaste.yaml')

In [ ]:
# EXP-02: DRAEM Perlin, No Freq Filter, rho_neg = 0.15
run_and_sync('exp02_draem', 'configs/experiments/exp02_draem.yaml')

In [ ]:
# EXP-03: NSA Poisson, No Freq Filter, rho_neg = 0.15
run_and_sync('exp03_nsa', 'configs/experiments/exp03_nsa.yaml')

In [ ]:
# EXP-04: CutPaste (Base) + 2D-FFT Notch, rho_neg = 0.15
run_and_sync('exp04_cutpaste_fft', 'configs/experiments/exp04_cutpaste_fft.yaml')

In [ ]:
# EXP-05: CutPaste (Base) + Haar 2D-DWT Boost, rho_neg = 0.15 + MVTec-AD-C (Table V +2D-DWT)
run_and_sync('exp05_cutpaste_dwt', 'configs/experiments/exp05_cutpaste_dwt.yaml')

In [ ]:
# EXP-06: DRAEM Perlin + 2D-FFT Notch, rho_neg = 0.15
run_and_sync('exp06_draem_fft', 'configs/experiments/exp06_draem_fft.yaml')

In [ ]:
# EXP-07: NSA Poisson + Haar 2D-DWT Boost, rho_neg = 0.15
run_and_sync('exp07_nsa_dwt', 'configs/experiments/exp07_nsa_dwt.yaml')

In [ ]:
# EXP-08: Full Inter-Domain System (Material-Adaptive: Texture=DRAEM+FFT | Object=NSA+DWT, rho_neg = 0.15)
run_and_sync('exp08_full_adaptive', 'configs/experiments/exp08_full_adaptive.yaml')

## 6. Consolidate Table III–V Summary & Sync to Drive

In [ ]:
# Restore any completed summaries from Drive before aggregating
import glob, os, shutil
for sf in glob.glob(os.path.join(DRIVE_RESULTS, '*', 'summary.json')):
    exp_id = os.path.basename(os.path.dirname(sf))
    os.makedirs(f'results/experiments/{exp_id}', exist_ok=True)
    shutil.copy2(sf, f'results/experiments/{exp_id}/summary.json')

!python experiments/ablation_matrix.py
!cp -f results/ablation_summary.csv "{DRIVE_RESULTS}/ablation_summary.csv"
!cp -f results/ablation_summary.md "{DRIVE_RESULTS}/ablation_summary.md"
with open('results/ablation_summary.md', 'r') as f:
    print(f.read())

## 7. Unified Single-GPU Latency Benchmark (All 5 Rows of Table VI)
Measures all 5 stages of **Table VI** on the **exact same GPU** (`YOLOv11l-seg@640`, matching Tables II–V):
1. PyTorch CPU (FP32)
2. PyTorch GPU (FP16)
3. TensorRT GPU (FP16) *(falls back to ONNX Runtime GPU if TensorRT is unavailable)*
4. Frequency Preprocessing (`RGB <-> YCrCb` + `cuFFT` Notch / Haar `2D-DWT`, FP32 GPU)
5. End-to-End (`T_freq` + TensorRT/Best Backbone)

In [ ]:
import json, os, time, torch, torch.nn.functional as F
from ultralytics import YOLO

assert torch.cuda.is_available(), 'CUDA GPU required for Table VI benchmark'
device = torch.device('cuda')
gpu_name = torch.cuda.get_device_name(0)
BENCH_IMGSZ = 640
DRIVE_RESULTS = globals().get('DRIVE_RESULTS', '/content/drive/MyDrive/industrial-defect-segmentation/results')
DRIVE_CKPTS = globals().get('DRIVE_CKPTS', '/content/drive/MyDrive/industrial-defect-segmentation/checkpoints')

# Use EXP-08 best.pt if available; otherwise yolo11l-seg.pt (identical architecture = identical latency)
exp08_ckpt = os.path.join(DRIVE_CKPTS, 'exp08_full_adaptive_best.pt')
bench_weights = exp08_ckpt if os.path.exists(exp08_ckpt) else 'yolo11l-seg.pt'
print(f'Benchmarking Table VI on {gpu_name} @ {BENCH_IMGSZ}x{BENCH_IMGSZ} using {bench_weights}')

def bench_cuda(fn, warmup=30, iters=150):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(iters):
            fn()
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) * 1000.0 / iters

# --- 1. PyTorch CPU (FP32) ---
cpu_model = YOLO(bench_weights).model.to('cpu').float().eval()
x_cpu = torch.rand((1, 3, BENCH_IMGSZ, BENCH_IMGSZ), dtype=torch.float32)
with torch.inference_mode():
    for _ in range(3):
        cpu_model(x_cpu)
    t0 = time.perf_counter()
    for _ in range(10):
        cpu_model(x_cpu)
    cpu_ms = round((time.perf_counter() - t0) * 1000.0 / 10, 2)
del cpu_model, x_cpu

# --- 2. PyTorch GPU (FP16) ---
gpu_model = YOLO(bench_weights).model.to(device).half().eval()
x_gpu_fp16 = torch.rand((1, 3, BENCH_IMGSZ, BENCH_IMGSZ), device=device, dtype=torch.float16)
pytorch_gpu_ms = round(bench_cuda(lambda: gpu_model(x_gpu_fp16)), 2)
del gpu_model, x_gpu_fp16
torch.cuda.empty_cache()

# --- 3. TensorRT GPU (FP16) via Ultralytics Export ---
trt_ms, trt_runtime = None, 'TensorRT FP16'
try:
    import importlib
    if importlib.util.find_spec('tensorrt') is None:
        !pip install -q tensorrt
    eng_path = YOLO(bench_weights).export(format='engine', imgsz=BENCH_IMGSZ, half=True, device=0, verbose=False)
    trt_yolo = YOLO(eng_path, task='segment')
    dummy_np = (torch.rand((BENCH_IMGSZ, BENCH_IMGSZ, 3)).numpy() * 255).astype('uint8')
    trt_ms = round(bench_cuda(lambda: trt_yolo.predict(dummy_np, imgsz=BENCH_IMGSZ, half=True, device=0, verbose=False), warmup=20, iters=100), 2)
except Exception as exc:
    print(f'[WARN] TensorRT export skipped ({exc}); using PyTorch FP16 as fallback.')
    trt_ms = pytorch_gpu_ms
    trt_runtime = 'PyTorch GPU FP16 (TRT fallback)'

# --- 4. Frequency Preprocessing on GPU (RGB <-> YCrCb + cuFFT Notch / Haar 2D-DWT, FP32) ---
x_rgb = torch.rand((1, 3, BENCH_IMGSZ, BENCH_IMGSZ), device=device, dtype=torch.float32)
notch_mask = torch.ones((1, 1, BENCH_IMGSZ, BENCH_IMGSZ), device=device, dtype=torch.float32)
h0 = torch.tensor([1.0, 1.0], device=device) / (2.0 ** 0.5)
h1 = torch.tensor([1.0, -1.0], device=device) / (2.0 ** 0.5)
dwt_w = torch.stack([torch.outer(h0, h0), torch.outer(h0, h1), torch.outer(h1, h0), torch.outer(h1, h1)], dim=0).unsqueeze(1)

def rgb_to_y(rgb):
    return 0.299 * rgb[:, 0:1] + 0.587 * rgb[:, 1:2] + 0.114 * rgb[:, 2:3]

def run_fft():
    y = rgb_to_y(x_rgb)
    f = torch.fft.fftshift(torch.fft.fft2(y)) * notch_mask
    y_rec = torch.fft.ifft2(torch.fft.ifftshift(f)).real
    return x_rgb + (y_rec - y)

def run_dwt():
    y = rgb_to_y(x_rgb)
    sub = F.conv2d(y, dwt_w, stride=2)
    sub[:, 0:1] *= 0.5
    sub[:, 1:] *= 1.6
    y_rec = F.conv_transpose2d(sub, dwt_w, stride=2)
    return x_rgb + (y_rec - y)

fft_ms = round(bench_cuda(run_fft), 3)
dwt_ms = round(bench_cuda(run_dwt), 3)
max_freq_ms = max(fft_ms, dwt_ms)
e2e_ms = round(trt_ms + max_freq_ms, 2)
e2e_fps = round(1000.0 / e2e_ms, 1)

lat_report = {
    'gpu': gpu_name,
    'resolution': f'{BENCH_IMGSZ}x{BENCH_IMGSZ}',
    'weights': bench_weights,
    'pytorch_cpu_fp32_ms': cpu_ms,
    'pytorch_cpu_fp32_fps': round(1000.0 / cpu_ms, 1),
    'pytorch_gpu_fp16_ms': pytorch_gpu_ms,
    'pytorch_gpu_fp16_fps': round(1000.0 / pytorch_gpu_ms, 1),
    'trt_runtime': trt_runtime,
    'trt_gpu_fp16_ms': trt_ms,
    'trt_gpu_fp16_fps': round(1000.0 / trt_ms, 1),
    'fft_cuFFT_ms': fft_ms,
    'fft_cuFFT_fps': round(1000.0 / fft_ms, 1),
    'dwt_haar_gpu_ms': dwt_ms,
    'dwt_haar_gpu_fps': round(1000.0 / dwt_ms, 1),
    'max_freq_ms': max_freq_ms,
    'max_freq_fps': round(1000.0 / max_freq_ms, 1),
    'end_to_end_ms': e2e_ms,
    'end_to_end_fps': e2e_fps,
    'viability': 'Inline ready' if e2e_fps >= 100.0 else 'Near real-time',
}
os.makedirs('results', exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)
for out_json in (
    os.path.join(DRIVE_RESULTS, 'table6_latency_benchmark.json'),
    os.path.join(DRIVE_RESULTS, 'gpu_frequency_latency.json'),
    'results/table6_latency_benchmark.json',
):
    with open(out_json, 'w') as f:
        json.dump(lat_report, f, indent=2)
print(json.dumps(lat_report, indent=2))

## 8. [Optional] Multi-Seed Variance Analysis (Seeds 1337 & 2026)
Run only after Section 5 completes if GPU quota permits.

In [ ]:
REPLICATE_SEEDS = [1337, 2026]
CORE_CONFIGS = [
    ('exp00_baseline',      'configs/experiments/exp00_baseline.yaml'),
    ('exp01_cutpaste',      'configs/experiments/exp01_cutpaste.yaml'),
    ('exp08_full_adaptive', 'configs/experiments/exp08_full_adaptive.yaml'),
]

for seed in REPLICATE_SEEDS:
    for exp_id, cfg_path in CORE_CONFIGS:
        run_id = f'{exp_id}_seed{seed}'
        drive_summary = os.path.join(DRIVE_RESULTS, run_id, 'summary.json')
        if os.path.exists(drive_summary):
            print(f'[SKIP] {run_id} already exists in Drive.')
            continue
        print(f'\n>>> Launching Replicate Run: {run_id} <<<')
        ret = subprocess.run(['python', 'experiments/run_experiment.py', '--config', cfg_path, '--seed', str(seed)], check=False)
        if ret.returncode != 0:
            raise RuntimeError(f'{run_id} failed with exit code {ret.returncode}')
        os.makedirs(os.path.join(DRIVE_RESULTS, run_id), exist_ok=True)
        shutil.copy2(f'results/experiments/{run_id}/summary.json', drive_summary)
        rep_csv = f'results/experiments/{run_id}/train_run/results.csv'
        if os.path.exists(rep_csv):
            shutil.copy2(rep_csv, os.path.join(DRIVE_RESULTS, run_id, 'results.csv'))
        shutil.rmtree(f'datasets/processed/{run_id}', ignore_errors=True)